# Rich Report

A compact executive-style analysis that combines narrative markdown, KPI tables, static charts, generated imagery, and sanitized HTML. It is designed to be the most complete single-command showcase.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import HTML, display

rng = np.random.default_rng(2026)
weeks = pd.date_range("2026-01-05", periods=12, freq="W-MON")
regions = ["north", "south", "west"]
rows = []
for region in regions:
    baseline = {"north": 420, "south": 360, "west": 390}[region]
    for index, week in enumerate(weeks):
        notebooks = baseline + index * rng.uniform(13, 22) + rng.normal(0, 18)
        media_mb = notebooks * rng.uniform(0.9, 1.4)
        saved_minutes = notebooks * rng.uniform(0.18, 0.32)
        rows.append((week, region, notebooks, media_mb, saved_minutes))

report = pd.DataFrame(rows, columns=["week", "region", "notebooks_viewed", "media_mb", "minutes_saved"])
latest = report[report["week"] == report["week"].max()].copy()
latest["notebooks_viewed"] = latest["notebooks_viewed"].round(0).astype(int)
latest["media_mb"] = latest["media_mb"].round(1)
latest["minutes_saved"] = latest["minutes_saved"].round(1)
display(latest[["region", "notebooks_viewed", "media_mb", "minutes_saved"]])

cards = "".join(
    f"<td style='padding:12px;border:1px solid #dbe3dc;background-color:#fbfcfb'><strong>{row.region.title()}</strong><br>{row.notebooks_viewed:,} notebooks<br>{row.minutes_saved:.1f} minutes saved</td>"
    for row in latest.itertuples(index=False)
)
display(HTML(f"<table style='border-collapse:collapse;width:100%;max-width:820px'><tr>{cards}</tr></table>"))


## Trend and mix

This section pairs a time-series trend with a region comparison, the sort of layout users often want to skim in a published notebook report.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.lineplot(data=report, x="week", y="notebooks_viewed", hue="region", marker="o", ax=axes[0])
axes[0].set_title("Viewed notebooks per week")
axes[0].set_xlabel("")
axes[0].set_ylabel("notebooks")
axes[0].tick_params(axis="x", rotation=35)

sns.barplot(data=latest, x="region", y="media_mb", palette=["#087f5b", "#1d4ed8", "#b45309"], ax=axes[1])
axes[1].set_title("Latest media served lazily")
axes[1].set_xlabel("")
axes[1].set_ylabel("media MB")
fig.tight_layout()
plt.show()


## Generated image artifact

The report includes a generated banner-like image to demonstrate non-chart image output without depending on external files.


In [ ]:
from PIL import Image, ImageDraw

image = Image.new("RGB", (860, 260), (247, 248, 246))
draw = ImageDraw.Draw(image)
draw.rectangle((28, 28, 832, 232), fill=(255, 255, 255), outline=(219, 227, 220), width=3)
for index, row in enumerate(latest.sort_values("notebooks_viewed", ascending=False).itertuples(index=False)):
    x0 = 70 + index * 250
    height = int(row.notebooks_viewed / latest["notebooks_viewed"].max() * 118)
    draw.rectangle((x0, 188 - height, x0 + 110, 188), fill=[(8,127,91), (29,78,216), (180,83,9)][index])
    draw.text((x0, 198), row.region.title(), fill=(23, 33, 28))
    draw.text((x0, 214), f"{row.notebooks_viewed:,} views", fill=(23, 33, 28))
draw.text((70, 54), "Notebook reading report", fill=(23, 33, 28))
draw.text((70, 78), "Generated offline and rendered as a lazy image asset", fill=(102, 116, 107))
display(image)


## Decision block

Sanitized HTML is useful for notebook summaries, callouts, and compact report sections.


In [ ]:
leader = latest.sort_values("minutes_saved", ascending=False).iloc[0]
html = f"""
<div style="border:1px solid #dbe3dc;background-color:#fbfcfb;padding:14px;max-width:820px">
  <strong>Recommendation:</strong>
  prioritize media-heavy notebook reviews in the {leader['region'].title()} region first.
  <br>
  <span style="color:#66746b">Reason: it produced {leader['minutes_saved']:.1f} estimated minutes saved in the latest week while serving {leader['media_mb']:.1f} MB of media through lazy assets.</span>
</div>
"""
display(HTML(html))
print("Rich report demo completed with deterministic offline data.")
